# NB 3.1 &mdash; Correlacions: quines variables van juntes

**MP 5134** &mdash; UT3

*Dades: California Housing, districtes censals de Califòrnia (cens de 1990).*

---
### Què farem avui

Al NB 2.2 vam afegir variables al model d'una en una i vam mirar quant baixava
l'error. Funcionava, però era una mica a cegues: per saber si una variable servia
de res, havíem d'entrenar un model sencer. Avui aprendrem a mirar-ho **abans
d'entrenar res**, amb un sol número per a cada parella de variables: la
**correlació**.

És un notebook curt i sense gaire codi nou. Quan acabis hauries de saber
respondre aquestes preguntes:

- Què vol dir que dues variables estan correlacionades, i què diu el número?
- Com es llegeix un mapa de calor?
- Quines variables de California Housing diuen gairebé el mateix?
- Quan convé fer servir Spearman en lloc de Pearson?
- Per què una correlació alta no vol dir que una cosa causi l'altra?

Pel camí trobaràs dues pauses **Ara tu**. Avui no hi ha exercicis finals: el que
aprenguis aquí ho faràs servir a la consolidació de la UT3.

## 1. Les dades de sempre

Tornem a California Housing, amb les mateixes línies del NB 2.2: carregar el
fitxer i apartar el test segellat amb el pany del `random_state=42`. Tot el que
mirarem avui és sobre `dades`, el 80% que ens queda.

Hi ha una importació nova, **seaborn**. És una biblioteca de gràfics construïda
damunt de matplotlib, i la farem servir per a una sola cosa: dibuixar el mapa de
calor de la secció 3. A Colab ja ve instal·lada.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# California Housing: districtes censals de Califòrnia (cens de 1990)
URL_DADES = ("https://raw.githubusercontent.com/ageron/handson-ml2/"
             "master/datasets/housing/housing.csv")

df = pd.read_csv(URL_DADES)

dades, test_segellat = train_test_split(df, test_size=0.2, random_state=42)
del test_segellat   # el pany del NB 2.2: no s'obre fins a la UT10

OBJECTIU = "median_house_value"
print(f"Districtes per treballar: {len(dades)}")

Els mateixos 16.512 districtes del NB 2.2. Recorda que cada fila és un districte
censal, no una casa, i que `median_house_value` és el preu mitjà dels habitatges
del districte: el que volem predir. L'hem desat a `OBJECTIU` per no haver
d'escriure el nom sencer cada vegada.

## 2. Què és una correlació

Al NB 2.2 vam veure que els ingressos (`median_income`) eren, de molt, la
variable que més ajudava a predir el preu. Comparem-la amb una altra que no
sembla tan prometedora, l'antiguitat dels habitatges (`housing_median_age`).
Dibuixem-les totes dues contra el preu, una al costat de l'altra, i calculem per
a cadascuna un número nou, la correlació, amb el mètode `corr()` de pandas.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for eix, variable in zip(eixos, ["median_income", "housing_median_age"]):
    r = dades[variable].corr(dades[OBJECTIU])
    eix.scatter(dades[variable], dades[OBJECTIU], alpha=0.05, s=8)
    eix.set_xlabel(variable)
    eix.set_title(f"Correlació amb el preu: {r:.2f}")

eixos[0].set_ylabel("Preu mitjà de l'habitatge ($)")
plt.show()

A l'esquerra, el núvol puja en diagonal: com més ingressos, més car és el
districte. No és una línia perfecta, però la tendència es veu d'una ullada. A la
dreta no hi ha cap tendència: districtes nous i vells hi tenen preus de tota
mena.

El número del títol, la **correlació**, resumeix això en una sola xifra. El que
calcula `corr()` per defecte és la **correlació de Pearson**, i funciona així:

- Va de **−1 a 1**.
- **El signe diu la direcció.** Positiva vol dir que quan una puja, l'altra també
  sol pujar. Negativa, que quan una puja, l'altra sol baixar.
- **La mida diu com de bé s'alineen els punts en una recta.** Un 1 o un −1 serien
  punts posats exactament en fila. Un 0 vol dir que no hi ha cap recta que els
  resumeixi.

Per llegir-la, ens servirà aquesta guia orientativa (no és cap llei, i cada camp
té els seus costums):

| Valor de la correlació (sense signe) | Com en diem |
|---|---|
| menys de 0,1 | no hi ha relació en línia recta |
| de 0,1 a 0,3 | feble |
| de 0,3 a 0,6 | moderada |
| més de 0,6 | forta |

Els ingressos, amb **0,69**, tenen una correlació forta amb el preu. L'antiguitat,
amb **0,10**, a penes en té. I tot això ho sabem sense haver entrenat cap model.

## 3. Totes les parelles alhora: el mapa de calor

Calcular les correlacions d'una en una seria pesat. Si cridem `corr()` sobre la
taula sencera, pandas les calcula **totes les parelles alhora** i en fa una
taula, la **matriu de correlació**. Com que tantes xifres juntes no es llegeixen
bé, la pintem com un **mapa de calor**: cada casella pren un color segons el
valor.

In [ ]:
matriu = dades.corr(numeric_only=True)

plt.figure(figsize=(9, 7))
sns.heatmap(matriu, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Matriu de correlació de California Housing")
plt.show()

Primer, com està feta la cel·la. `numeric_only=True` diu a pandas que només faci
servir les columnes numèriques: `ocean_proximity` és text i queda fora. Convertir
el text en números és el tema del NB 3.2. A `sns.heatmap`, `annot=True` escriu el
número a cada casella, `fmt=".2f"` el deixa amb dos decimals, `cmap="coolwarm"`
tria els colors (vermell per a les positives, blau per a les negatives) i
`vmin=-1, vmax=1` fixa l'escala perquè el blanc caigui just al zero.

Ara, com es llegeix el mapa:

- **La diagonal és tota d'uns**: cada variable va perfectament lligada amb ella
  mateixa. No diu res.
- **És simètric**: la casella de *A amb B* i la de *B amb A* són la mateixa
  correlació. N'hi ha prou de mirar la meitat.
- **Els colors forts són el que importa.** El blanc vol dir correlació gairebé
  nul·la.

I hi ha dues coses que salten a la vista. La primera, **un bloc vermell intens**
al mig, amb quatre variables (`total_rooms`, `total_bedrooms`, `population` i
`households`) que tenen entre elles correlacions de 0,86 a 0,98. Hi tornarem a la
secció 5. La segona, **un −0,92 entre la latitud i la longitud**. Recorda el mapa
del NB 2.2: Califòrnia és una franja inclinada, i com més al nord és un
districte, més a l'oest sol ser. No és cap descobriment sobre els habitatges, és
la forma de l'estat.

L'última fila (o l'última columna, que és el mateix) és la que ens interessa per
predir: **la correlació de cada variable amb el preu.**

## 4. Quines variables van més lligades al preu?

Traiem aquesta última columna de la matriu i ordenem-la. Volem les correlacions
més fortes al capdamunt, siguin positives o negatives, i per això ordenem pel
valor **sense signe**: `key=abs` vol dir «ordena segons el valor absolut».

In [ ]:
correlacio_preu = matriu[OBJECTIU].drop(OBJECTIU)
correlacio_preu.sort_values(key=abs, ascending=False).round(2)

`drop(OBJECTIU)` treu el preu de la llista, perquè la seva correlació amb ell
mateix (l'1 de la diagonal) no ens diu res.

El rànquing és clar: **els ingressos, amb 0,69, van molt per davant de tota la
resta**. La segona variable, la latitud, ja només té −0,14. Si haguéssim de
triar una sola entrada per al model, seria aquesta, i és la que vam triar
d'intuïció al NB 2.2.

Ara, compte amb una temptació: fer servir aquesta llista per llençar totes les
variables de baix. Fixa't en la **longitud**: té una correlació de −0,05, de les
més baixes de la llista. I en canvi, al NB 2.2, era la variable que més baixava
l'error després dels ingressos. Com pot ser?

Doncs perquè la correlació té dues limitacions que convé recordar sempre:

- **Mira les variables d'una en una.** La longitud, tota sola, no diu gaire del
  preu. Juntament amb la latitud, marca on és la costa, i la costa és cara.
- **Només veu rectes.** Si la relació fa corbes (com les dues muntanyes de la
  latitud al final del NB 2.2), la correlació la pot donar per gairebé nul·la.

Per això la correlació és un **bon primer filtre**, però no un veredicte. Les
variables amb correlació alta són candidates segures. Les de correlació baixa no
s'han de llençar sense provar-les, i qui decideix, com sempre, és el conjunt de
validació.

---
### Ara tu · Exercici 1: totals o proporcions?

Al NB 2.3 vam fabricar dues columnes noves a partir dels totals del districte:
les habitacions per llar i el percentatge de dormitoris. Fes-ho ara sobre
`dades`:

- `habitacions_per_llar`: `total_rooms` dividit entre `households`.
- `pct_dormitoris`: 100 per `total_bedrooms` dividit entre `total_rooms`.

Calcula la correlació de cadascuna amb el preu i compara-la amb la de
`total_rooms` i `total_bedrooms`, que ja tens a la llista de dalt. Quina diu més
del preu, el total o la proporció? Per què creus que passa?

Pista: `dades["columna_nova"] = dades["a"] / dades["b"]` crea la columna, i
`dades["columna_nova"].corr(dades[OBJECTIU])` en dona la correlació. Pensa en què
mesura de veritat un total de districte: la casa, o la mida del districte?

In [ ]:
# Escriu aquí el teu codi

## 5. Variables que diuen el mateix: la colinealitat

Tornem al bloc vermell del mapa de calor. Mirem-ne la parella més forta, el
nombre de llars i el nombre de dormitoris del districte.

In [ ]:
r = dades["households"].corr(dades["total_bedrooms"])

plt.figure(figsize=(6, 5))
plt.scatter(dades["households"], dades["total_bedrooms"], alpha=0.1, s=8)
plt.xlabel("households (llars del districte)")
plt.ylabel("total_bedrooms (dormitoris del districte)")
plt.title(f"Correlació: {r:.2f}")
plt.show()

És gairebé una línia: **0,98**. Té tota la lògica: cada llar té, més o menys,
un parell de dormitoris, així que si saps quantes llars hi ha en un districte,
saps quants dormitoris té. Les quatre variables del bloc vermell mesuren, en el
fons, **la mateixa cosa: com és de gran el districte**.

Quan dues variables d'entrada van tan juntes es diu que hi ha **colinealitat**, i
per al model és un problema. No tant per l'error, que gairebé no canvia, sinó
perquè el model **no sap a quina de les dues donar-li el mèrit**, i els
coeficients es tornen absurds. Ja ho vas viure a l'exercici 3 del NB 2.2: el
coeficient de `total_rooms` canviava completament en treure'n `total_bedrooms` i
`households`. A més, cada variable repetida és una columna més que no aporta res
de nou: al NB 2.2, afegir `households` quan el model ja tenia `total_bedrooms`
deixava l'error pràcticament igual.

Amb una matriu petita els blocs es veuen a ull, però amb moltes variables convé
que ho busqui l'ordinador. La cel·la següent fa una llista de totes les parelles
amb una correlació de més de 0,9, positiva o negativa.

In [ ]:
# Cada parella surt dues vegades a la matriu (A-B i B-A). Ens quedem només amb
# el triangle de dalt, sense la diagonal, i passem la matriu a una llista.
triangle_de_dalt = np.triu(np.ones(matriu.shape, dtype=bool), k=1)
parelles = matriu.where(triangle_de_dalt).stack()

parelles[parelles.abs() > 0.9].sort_values(key=abs, ascending=False).round(2)

No cal que entenguis cada peça de la cel·la; n'hi ha prou de saber què fa i
tenir-la a mà per reutilitzar-la. `np.triu` marca el triangle de dalt de la
matriu, `where` esborra la resta i `stack` passa la taula a una llista de
parelles.

Surten **cinc parelles**. Quatre són del bloc de la mida del districte. Per a
aquestes, la recepta pràctica és **quedar-se'n una**, o, millor encara,
**combinar-les en una proporció** que mesuri una altra cosa, com les habitacions
per llar de l'exercici 1. Ho farem al NB 3.2.

La cinquena és la latitud i la longitud, i aquí la recepta no s'aplica: ja hem
vist que juntes donen informació que cap de les dues no dona tota sola. Una
correlació alta entre dues entrades és **un avís per mirar-s'ho, no una ordre
per esborrar-ne una**.

## 6. Quan Pearson s'equivoca: Spearman

Fabriquem una altra columna: les **persones per llar**, la població del districte
dividida entre el nombre de llars. La intuïció diu que als districtes on les llars
són més plenes (més gent vivint junta) els habitatges deuen ser més barats.
Calculem-ne la correlació amb el preu de dues maneres: amb Pearson, com fins ara,
i amb **Spearman**, que és l'altra correlació que porta pandas.

In [ ]:
dades["persones_per_llar"] = dades["population"] / dades["households"]

pearson = dades["persones_per_llar"].corr(dades[OBJECTIU])
spearman = dades["persones_per_llar"].corr(dades[OBJECTIU], method="spearman")

print(f"Pearson:  {pearson:.2f}")
print(f"Spearman: {spearman:.2f}")

Dues respostes molt diferents. Pearson diu **−0,02**: cap relació. Spearman diu
**−0,26**: una relació negativa feble, en la direcció que esperàvem. Quina té
raó? Quan dues correlacions no coincideixen, només hi ha una manera de saber-ho:
**dibuixar els punts**.

In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(dades["persones_per_llar"], dades[OBJECTIU], alpha=0.1, s=8)
plt.xlabel("Persones per llar")
plt.ylabel("Preu mitjà de l'habitatge ($)")
plt.title("Persones per llar contra preu")
plt.show()

Gairebé no es veu res: tots els districtes estan aixafats contra la vora
esquerra, i uns pocs punts solts s'escampen cap a la dreta fins a més de **1.200
persones per llar**. Cap habitatge normal no és així; segurament són districtes
amb molt poques llars registrades i molta gent que no hi viu en una casa
normal. Amb més de 10 persones per llar només n'hi ha 35, de més de setze mil
districtes, però són tan extrems que **s'enduen tota la correlació de Pearson**,
que fa servir els valors tal com són.

**Spearman** s'ho mira d'una altra manera: no fa servir els valors, sinó **la
posició de cada districte quan els ordenes**. El districte amb més persones per
llar és l'últim de la fila, tant si en té 12 com si en té 1.200. Així, els
valors extrems no poden estirar el resultat. I com que només mira l'ordre,
Spearman detecta qualsevol relació que vagi sempre en la mateixa direcció, encara
que no sigui una recta: el que s'anomena una relació **monòtona**.

Fem un zoom a la zona on viu gairebé tothom, entre 1 i 6 persones per llar.

In [ ]:
zona_normal = dades[dades["persones_per_llar"].between(1, 6)]
r_zoom = zona_normal["persones_per_llar"].corr(zona_normal[OBJECTIU])

plt.figure(figsize=(8, 4))
plt.scatter(zona_normal["persones_per_llar"], zona_normal[OBJECTIU], alpha=0.1, s=8)
plt.xlabel("Persones per llar")
plt.ylabel("Preu mitjà de l'habitatge ($)")
plt.title(f"Només districtes d'entre 1 i 6 persones per llar (Pearson: {r_zoom:.2f})")
plt.show()

print(f"Districtes fora del zoom: {len(dades) - len(zona_normal)}")

Ara sí que es veu la tendència: **com més persones per llar, més barat és el
districte**. I no és una recta: entre 1,5 i 4,5 persones per llar, el preu mitjà
baixa gairebé a la meitat (de 274.000 \$ a 140.000 \$), i a partir d'aquí
s'aplana. La ratlla de punts de dalt de tot són els preus tallats a 500.001 \$
que ja coneixes del NB 2.2. Sense els districtes extrems, Pearson ja dona
**−0,28**, molt a prop dels −0,26 de Spearman.

La lliçó pràctica:

- **Calcula sempre les dues.** Costa una línia més.
- **Si coincideixen**, pots estar tranquil: la relació és prou recta i no hi ha
  valors extrems que manin.
- **Si no coincideixen**, dibuixa els punts. Gairebé sempre hi trobaràs valors
  extrems o una corba. Com tractar-los és feina del NB 3.2.

---
### Ara tu · Exercici 2: Pearson contra Spearman, totes alhora

`dades.corr(numeric_only=True, method="spearman")` calcula la matriu de
correlació de Spearman sencera.

- Dibuixa-la amb un mapa de calor, igual que el de la secció 3. A primer cop
  d'ull, s'assembla a la de Pearson?
- Fes una taula amb dues columnes, la correlació de Pearson i la de Spearman de
  cada variable amb el preu, i una tercera amb la diferència entre totes dues
  (sense signe).
- Quina variable, a part de `persones_per_llar`, té la diferència més gran?
  Dibuixa-la contra el preu i explica per què creus que no coincideixen.

Pista: `pd.DataFrame({"Pearson": serie_1, "Spearman": serie_2})` posa dues
columnes una al costat de l'altra, i `(taula["Pearson"] - taula["Spearman"]).abs()`
en dona la diferència sense signe.

In [ ]:
# Escriu aquí el teu codi

## 7. Correlació no és causalitat

Una última idea, sense codi, però potser la més important del notebook.

Als estius es venen més gelats i també hi ha més ofegats a les platges. Les dues
coses tenen una correlació fortíssima. Però els gelats no ofeguen ningú: és la
**calor** la que fa que es venguin més gelats i que més gent vagi a nedar. Quan
dues variables van juntes, pot ser que una causi l'altra, però també pot ser que
**totes dues depenguin d'una tercera** que no estem mirant.

A les nostres dades en tenim un exemple a la vista. `total_rooms` i
`population` tenen una correlació de 0,86, i no és que les habitacions portin
gent: és que tots dos són grans quan el districte és gran. I els ingressos i el
preu, amb 0,69? Guanyar més no fa que la teva casa valgui més. Més aviat és al
revés: qui guanya més pot pagar per viure on les cases són cares.

**Per predir, això no importa gaire**: al model li basta que la variable vagi
junta amb l'objectiu. **Per decidir, importa molt.** Si un ajuntament volgués
abaratir l'habitatge baixant els sous del barri, s'equivocaria de mig a mig.

I encara hi ha un cas pitjor: les **correlacions espúries**, parelles de
variables que van juntes **per pura casualitat**, sense cap relació real. Com més
variables mires, més fàcil és trobar-ne alguna. A la consolidació de la UT3 n'hi
haurà una d'amagada.

## 8. El que ens emportem

Avui no hem entrenat cap model, però hem après moltes coses sobre les variables
abans de fer-ho. Aquestes són les decisions que en traiem, i que el NB 3.2
posarà en pràctica:

| Variable | Correlació amb el preu | Què en fem |
|---|---|---|
| `median_income` | 0,69 | entrada segura: és, de molt, la més lligada al preu |
| `latitude`, `longitude` | −0,14 i −0,05 | es queden, totes dues juntes: soles diuen poc, juntes marquen la costa |
| `housing_median_age` | 0,10 | feble; es queda i que decideixi la validació |
| `total_rooms`, `total_bedrooms`, `population`, `households` | entre −0,03 i 0,13 | colinealitat: les substituirem per proporcions |
| `persones_per_llar` | −0,02 (Pearson), −0,26 (Spearman) | candidata nova, però amb valors extrems per tractar |
| `ocean_proximity` | (és text) | encara no la podem fer servir: NB 3.2 |

Aquesta taula és la llavor d'un document que faràs a la consolidació: la **fitxa
de variables**, amb les entrades, la sortida i les correlacions trobades. La
tornarem a necessitar a la **UT5**, quan els arbres de decisió ens diguin quines
variables els han estat més útils i les puguem comparar amb aquestes
correlacions.